# 🌿 Olive Yield Across Mediterranean Countries: Environmental and Socioeconomic Factors

## Notebook 01: Data Collection and Consolidation

## Overview

Olive production is associated with climate, soil conditions, land use, agricultural resources, and the broader socioeconomic context. This project examines these factors across 11 Mediterranean olive-producing countries over the period 2000–2024.

This notebook develops the **Bronze and Silver data layers** of the project. It collects, processes, validates, and integrates agricultural, climate, socioeconomic, land-use, and temperature-change data from authoritative public sources.

All datasets are harmonized to a common **Country–Year** structure, where each observation represents one country in one year. The resulting enriched Silver dataset provides the validated input for Gold-layer feature engineering in Notebook 2.

## Data Sources and Variables

### FAOSTAT: Agricultural Data

Annual agricultural indicators:

- olive production (tonnes);
- harvested area (ha);
- olive yield (kg/ha).

Source: https://www.fao.org/faostat/

### ERA5: Climate Data

ERA5 data from the Copernicus Climate Data Store are spatially processed for the selected countries and aggregated into annual Country–Year indicators:

- air temperature (°C);
- total precipitation (mm);
- solar radiation (MJ/m²);
- dew-point temperature (°C);
- wind speed (m/s);
- soil moisture at 0–7 cm (m³/m³);
- soil moisture at 7–28 cm (m³/m³);
- soil moisture at 28–100 cm (m³/m³);
- soil temperature at 7–28 cm (°C);
- soil temperature at 28–100 cm (°C).

Source: https://cds.climate.copernicus.eu/

### World Bank: Socioeconomic and Land-Use Data

Annual socioeconomic and land-use indicators:

- population (people);
- GDP per capita (current US$);
- agricultural land (% of total land area);
- forest area (% of total land area);
- rural population (% of total population);
- agriculture value added (% of GDP).

Source: https://data.worldbank.org/

### FAOSTAT: Temperature-Change Indicators

FAOSTAT Environmental Indicators provide annual temperature-change values (°C).

For each country and year, the indicator represents the difference between the observed temperature and the country’s average temperature during the **1951–1980 reference period**:

- positive values indicate temperatures above the historical baseline;
- negative values indicate temperatures below the historical baseline;
- values close to zero indicate temperatures near the historical baseline.

The annual temperature-change indicator is integrated into the consolidated Silver dataset as an additional climate feature. It is distinct from the ERA5 mean air temperature:

- **ERA5 mean air temperature (°C):** the average temperature observed for a country in a given year;
- **FAOSTAT temperature change (°C):** the deviation of that year’s temperature from the country’s 1951–1980 reference average.

Source: https://www.fao.org/faostat/


## Data Consolidation

The source datasets are processed through the Bronze and Silver stages of the project’s Medallion Architecture:

```text
FAOSTAT agricultural data ──────┐
                                │
ERA5 climate data ──────────────┼──► Standardized source datasets
                                │
World Bank indicators ──────────┘
                                              │
                                              ▼
                                  Consolidated Silver dataset
                                              │
FAOSTAT temperature change ───────────────────┤
                                              ▼
                                    Enriched Silver dataset
```

The consolidation process:

1. filters the source data to the selected countries and study period;
2. harmonizes country names, years, variables, units, and data types;
3. aggregates ERA5 climate data to the Country–Year level;
4. validates temporal coverage, missing values, and Country–Year uniqueness;
5. integrates the agricultural, climate, socioeconomic, and land-use datasets using `Country` and `Year`;
6. enriches the consolidated dataset with annual temperature change relative to the 1951–1980 baseline;
7. exports the validated datasets for downstream processing.


## Notebook Outputs

This notebook produces:

- Raw, source-aligned Bronze dataset structures;
- standardized agricultural, climate, and socioeconomic Silver datasets;
- a consolidated Country–Year Silver dataset;
- an enriched Silver dataset containing the annual temperature-change indicator.

The enriched Silver dataset is the primary output of this notebook and the input for Gold-layer feature engineering in Notebook 02.

## Imports and Display Settings

In [1]:
# File and archive handling
from io import BytesIO
from pathlib import Path
from zipfile import ZipFile

# Climate data access
import cdsapi

# Geospatial processing
from global_land_mask import globe

# Data manipulation
import numpy as np
import pandas as pd
import xarray as xr

# Web requests
import requests

## Project Configuration
This section defines the project directories, the study countries, and the country name harmonization.

In [ ]:
# Project directories
BASE_DATA_DIR = Path("../data")

BRONZE_DIR = BASE_DATA_DIR / "01_bronze"
SILVER_DIR = BASE_DATA_DIR / "02_silver"
ERA5_EXTRACTED_DIR = BRONZE_DIR / "era5_olives_extracted"

# Create directories if they do not already exist
BRONZE_DIR.mkdir(parents=True, exist_ok=True)
SILVER_DIR.mkdir(parents=True, exist_ok=True)
ERA5_EXTRACTED_DIR.mkdir(parents=True, exist_ok=True)

# Study countries
COUNTRIES_ISO3 = {
    "ESP": "Spain",
    "ITA": "Italy",
    "GRC": "Greece",
    "PRT": "Portugal",
    "MAR": "Morocco",
    "TUN": "Tunisia",
    "DZA": "Algeria",
    "EGY": "Egypt",
    "TUR": "Turkiye",
    "LBY": "Libya",
    "SYR": "Syria"
}

# Country name standardization
COUNTRY_ALIASES = {
    "Turkey": "Turkiye",
    "Türkiye": "Turkiye",
    "TÃ¼rkiye": "Turkiye",
    "Turkiye": "Turkiye",
    "Syrian Arab Republic": "Syria"
}

def normalize_country_name(name):
    """Standardize country names across data sources."""
    
    if pd.isna(name):
        return name

    name = str(name).strip()

    return COUNTRY_ALIASES.get(name, name)

# Validation
print("Directories initialized successfully!")

Directories initialized successfully!


## FAOSTAT Data Collection

Agricultural data are retrieved from the FAOSTAT bulk repository and filtered to retain olive-related observations for the selected Mediterranean countries.  

The resulting dataset forms the agricultural Bronze layer of the project and serves as the foundation for subsequent validation, cleaning, and integration steps.

In [3]:
# FAOSTAT data download
FAOSTAT_BULK_URL = (
    "https://fenixservices.fao.org/faostat/static/bulkdownloads/"
    "Production_Crops_Livestock_E_All_Data_(Normalized).zip"
)

response = requests.get(FAOSTAT_BULK_URL, timeout=120)
response.raise_for_status()

# Load dataset
with ZipFile(BytesIO(response.content)) as zf:
    csv_name = zf.namelist()[0]

    faostat_raw = pd.read_csv(
        zf.open(csv_name),
        encoding="latin1",
        low_memory=False)
    
# Harmonize country names
faostat_raw["Country"] = faostat_raw["Area"].map(normalize_country_name)

# Filter the item olive for the selected countries
faostat_bronze = faostat_raw[
    (faostat_raw["Item"] == "Olives")
    & (faostat_raw["Country"].isin(COUNTRIES_ISO3.values()))
].copy()

# Validation
print(f"FAOSTAT Bronze dataset shape: {faostat_bronze.shape}")
display(faostat_bronze.head())

FAOSTAT Bronze dataset shape: (2011, 15)


,Area Code,Area Code (M49),Area,Item Code,Item Code (CPC),Item,Element Code,Element,Year Code,Year,Unit,Value,Flag,Note,Country
43727,4,'012,Algeria,260,'01450,Olives,5312,Area harvested,1961,1961,ha,0.0,A,NaN,Algeria
43728,4,'012,Algeria,260,'01450,Olives,5312,Area harvested,1962,1962,ha,0.0,A,NaN,Algeria
43729,4,'012,Algeria,260,'01450,Olives,5312,Area harvested,1963,1963,ha,0.0,A,NaN,Algeria
43730,4,'012,Algeria,260,'01450,Olives,5312,Area harvested,1964,1964,ha,0.0,A,NaN,Algeria
43731,4,'012,Algeria,260,'01450,Olives,5312,Area harvested,1965,1965,ha,0.0,A,NaN,Algeria


The FAOSTAT bulk dataset was successfully downloaded and filtered to retain olive-related observations for the selected Mediterranean countries.

The resulting Bronze dataset contains:  
- **2,011 rows**
- **15 columns**

A preview of the first records confirms that the dataset includes agricultural indicators such as production, harvested area, and yield for the selected study countries.  

At this stage, the data remains in the Bronze layer, meaning that only source-specific filtering and country harmonization have been applied. The original FAOSTAT structure is preserved to maintain data traceability.  

Further cleaning, standardization, and reshaping will be performed before integrating the agricultural data with climate and socio-economic indicators in the Silver layer.

## FAOSTAT Bronze Dataset quality checks

This section performs a comprehensive initial data profiling of the faostat_bronze DataFrame by displaying its dimensions, column names, missing and duplicate values, and data distribution across countries and years.

In [4]:
print(
    f"Dataset shape: {faostat_bronze.shape}\n"
    f"\nColumns:\n{faostat_bronze.columns.tolist()}\n"
    f"\nMissing values:\n{faostat_bronze.isna().sum().sort_values(ascending=False)}\n"
    f"\nDuplicate rows: {faostat_bronze.duplicated().sum()}\n"
    f"\nCountries included:\n{sorted(faostat_bronze['Country'].dropna().unique())}\n"
    f"\nYears covered: {faostat_bronze['Year'].min()} - {faostat_bronze['Year'].max()}\n"
    f"\nRows per country:\n{faostat_bronze['Country'].value_counts().sort_index()}"
)

Dataset shape: (2011, 15)

Columns:
['Area Code', 'Area Code (M49)', 'Area', 'Item Code', 'Item Code (CPC)', 'Item', 'Element Code', 'Element', 'Year Code', 'Year', 'Unit', 'Value', 'Flag', 'Note', 'Country']

Missing values:
Note               1968
Value                91
Area Code             0
Item Code             0
Item Code (CPC)       0
Area Code (M49)       0
Area                  0
Element Code          0
Item                  0
Element               0
Year Code             0
Unit                  0
Year                  0
Flag                  0
Country               0
dtype: int64

Duplicate rows: 0

Countries included:
['Algeria', 'Egypt', 'Greece', 'Italy', 'Libya', 'Morocco', 'Portugal', 'Spain', 'Syria', 'Tunisia', 'Turkiye']

Years covered: 1961 - 2024

Rows per country:
Country
Algeria     184
Egypt       192
Greece      166
Italy       192
Libya       168
Morocco     192
Portugal    168
Spain       173
Syria       192
Tunisia     192
Turkiye     192
Name: count, dtype

The validation confirms that the dataset provides the selected Mediterranean countries.

No duplicate records were detected. Missing values are limited to the `Value` column and represent a small proportion of the dataset. These records will be investigated during the Silver-layer transformation stage to determine whether treatment or exclusion is required.

Overall, the Bronze dataset is structurally consistent and suitable for further processing and integration with climate and socio-economic data sources.

### Agricultural Measurement Units check

This section review the measurement units associated with each agricultural indicator to ensure consistency before dataset transformation.

In [5]:
display(
    faostat_bronze[["Element", "Unit"]]
    .drop_duplicates()
    .sort_values("Element"))

,Element,Unit
43727,Area harvested,ha
43847,Production,t
43791,Yield,kg/ha


### Export Bronze Dataset
Save the validated FAOSTAT Bronze dataset for subsequent processing and Silver-layer transformation.

In [6]:
faostat_bronze.to_csv(
    BRONZE_DIR / "faostat_olives_bronze.csv",
    index=False,
    encoding="utf-8"
)

The FAOSTAT Bronze dataset was successfully exported to the project's Bronze layer.


## FAOSTAT Silver Transformation

The Bronze dataset follows the original FAOSTAT structure, where agricultural indicators are stored as separate records.

To facilitate integration with climate and socio-economic datasets, the data is reshaped into a standardized Country-Year format. Each agricultural indicator becomes a dedicated feature associated with a specific country and year.

The resulting dataset forms the FAOSTAT Silver layer and will later be integrated with climate and socio-economic information.

In [7]:
# Select the agricultural indicators required for the analysis

keep_elements = [
    "Area harvested",
    "Production",
    "Yield"
]

faostat_filtered = faostat_bronze[
    faostat_bronze["Element"].isin(keep_elements)
].copy()

# Transform the dataset into a Country-Year structure

faostat_silver = (
    faostat_filtered
    .pivot_table(
        index=["Country", "Year"],
        columns="Element",
        values="Value",
        aggfunc="first"
    )
    .reset_index()
    .rename(columns={
        "Area harvested": "Area_Harvested_ha",
        "Production": "Production_tonnes",
        "Yield": "Yield_kg_per_ha"
    })
    .sort_values(["Country", "Year"])
    .reset_index(drop=True)
)


print(f"FAOSTAT Silver dataset shape: {faostat_silver.shape}")

display(faostat_silver.head())

FAOSTAT Silver dataset shape: (704, 5)


Element,Country,Year,Area_Harvested_ha,Production_tonnes,Yield_kg_per_ha
0,Algeria,1961,0.0,160000.0,NaN
1,Algeria,1962,0.0,143000.0,NaN
2,Algeria,1963,0.0,148300.0,NaN
3,Algeria,1964,0.0,148559.0,NaN
4,Algeria,1965,0.0,153169.0,NaN


The transformation converts the original FAOSTAT long-format structure into a tabular dataset with a standardized Country-Year structure, suitable for integration, analysis, and modeling.

The resulting dataset contains the following variables:  
- Country
- Year
- Area_Harvested_ha
- Production_tonnes
- Yield_kg_per_ha

## FAOSTAT Silver Dataset Validation

This section represnts a second validation stage: assess data completeness, verify the uniqueness of Country-Year records, and ensure that the resulting dataset is ready for integration with climate and socio-economic data source. 

In [8]:
# Dataset dimensions, missing values and duplicates
print(
    f"Dataset shape: {faostat_silver.shape}\n"
    f"\nMissing values:"
)
display(faostat_silver.isna().sum())

print(f"\nDuplicate Country-Year rows: {faostat_silver.duplicated(subset=['Country', 'Year']).sum()}")

# Remove records with missing target values
faostat_silver = faostat_silver.dropna(subset=["Yield_kg_per_ha"])

# Post-processing display
print("\nMissing values after processing:")
display(faostat_silver.isna().sum())
display(faostat_silver.head())


Dataset shape: (704, 5)

Missing values:


Element
Country                0
Year                   0
Area_Harvested_ha     91
Production_tonnes      1
Yield_kg_per_ha      100
dtype: int64


Duplicate Country-Year rows: 0

Missing values after processing:


Element
Country              0
Year                 0
Area_Harvested_ha    0
Production_tonnes    0
Yield_kg_per_ha      0
dtype: int64

Element,Country,Year,Area_Harvested_ha,Production_tonnes,Yield_kg_per_ha
8,Algeria,1969,97260.0,118902.0,1222.5
9,Algeria,1970,98840.0,137571.0,1391.9
10,Algeria,1971,125090.0,167836.0,1341.7
11,Algeria,1972,146300.0,171550.0,1172.6
12,Algeria,1973,151130.0,119061.0,787.8


The validation confirms that the Silver dataset provides a complete and standardized Country-Year representation of olive production data.

No duplicate Country-Year records were detected, ensuring the uniqueness of each observation.  
Records with missing yield values were removed to guarantee data completeness prior to integration with climate and socio-economic datasets.  
The validated dataset is now ready for the subsequent integration and enrichment stages of the pipeline.

In [9]:
# Save Silver dataset

faostat_silver.to_csv(
    SILVER_DIR / "faostat_olives_silver.csv",
    index=False,
    encoding="utf-8"
)

## ERA5 Climate Data Collection

To characterize the environmental conditions related to olive cultivation, climate data are retrieved from the Copernicus Climate Data Store (ERA5 atmospheric reanalysis data).

The selected variables capture key dimensions of agricultural production:

- Air temperature
- Precipitation
- Atmospheric humidity
- Wind speed
- Solar radiation
- Soil moisture
- Soil temperature

Monthly climate data are collected for the Mediterranean region covering the selected olive-producing countries.

The downloaded dataset spans the period **1985–2024**, allowing the construction of long-term climate indicators and historical trends.

The resulting dataset forms the climate component of the Bronze layer and will later be aggregated into annual indicators within the Silver layer.

> **Note:** Reproducing this step requires a Copernicus Climate Data Store (CDS) account and valid API credentials configured locally.

In [ ]:
# ERA5 Climate Data Download

ERA5_ZIP_PATH = BASE_DIR / ".." / "data" / "01_bronze" / "era5_olives.zip"

if not ERA5_ZIP_PATH.exists():

    client = cdsapi.Client()
    request = {
        "product_type": ["monthly_averaged_reanalysis"],

        "variable": [
            # Atmospheric conditions
            "2m_temperature",
            "2m_dewpoint_temperature",
            "10m_wind_speed",
            "total_precipitation",

            # Solar energy
            "surface_net_solar_radiation",

            # Soil moisture
            "volumetric_soil_water_layer_1",
            "volumetric_soil_water_layer_2",
            "volumetric_soil_water_layer_3",

            # Soil temperature
            "soil_temperature_level_2",
            "soil_temperature_level_3"
        ],

        "year": [str(year) for year in range(1985, 2025)],

        "month": [
            "01", "02", "03", "04", "05", "06",
            "07", "08", "09", "10", "11", "12"
        ],

        "time": ["00:00"],

        # Mediterranean region
        # North, West, South, East
        "area": [46, -14, 25, 45],

        "data_format": "netcdf",
        "download_format": "unarchived"
    }

    client.retrieve(
        "reanalysis-era5-single-levels-monthly-means",
        request
    ).download(str(ERA5_ZIP_PATH))

    print("ERA5 dataset successfully downloaded.")

else:
    print("ERA5 dataset already available:", ERA5_ZIP_PATH)

ERA5 dataset already available: era5_olives.zip


### ERA5 Download Validation

This section verifies the succesful download and local storage of the dataset before proceeding with extraction and processing.

In [11]:
print(f"ERA5 file exists: {ERA5_ZIP_PATH.exists()}")

if ERA5_ZIP_PATH.exists():
    file_size_mb = ERA5_ZIP_PATH.stat().st_size / (1024 ** 2)

    print(f"File size (MB): {file_size_mb:.2f}")

ERA5 file exists: True
File size (MB): 153.88


## ERA5 Data Extraction

The ERA5 climate dataset was downloaded as a compressed archive containing NetCDF files.
This section extracts the archive and identifies the available NetCDF files that will be used for subsequent climate data processing and aggregation.

In [12]:
# ERA5 Climate Data Extraction
ERA5_ZIP_PATH = BASE_DIR / "era5_olives.zip"

if ERA5_ZIP_PATH.exists():
    with ZipFile(ERA5_ZIP_PATH, "r") as zf:
        zf.extractall(ERA5_EXTRACTED_DIR)
    print("ERA5 dataset successfully extracted.")
else:
    print(
        "ERA5 ZIP file not found. Run the download step "
        "or place 'era5_olives.zip' in the project directory."
    )

# Locate extracted NetCDF files
nc_files = sorted(ERA5_EXTRACTED_DIR.glob("*.nc"))
print(f"Total NetCDF files ready: {len(nc_files)}")


ERA5 dataset successfully extracted.
Total NetCDF files ready: 2


The ERA5 archive was successfully extracted into the project directory and two NetCDF files were identified. 

## ERA5 Dataset Inspection

Before merging the extracted NetCDF files, it is important to inspect their structure and verify that all requested climate variables were successfully downloaded.

This step helps identify the variable names used by ERA5 and ensures consistency with the subsequent processing workflow.

In [13]:
for file in nc_files:
        if file.exists():
            # Standard open call - automatically uses netcdf4 now that paths are clean
            with xr.open_dataset(file) as ds_part:
                print(f"\nFile: {file.name}")
                print("Variables:")
                for var in ds_part.data_vars:
                    print(f" - {var}")


File: data_stream-moda_stepType-avgad.nc
Variables:
 - tp
 - ssr

File: data_stream-moda_stepType-avgua.nc
Variables:
 - t2m
 - d2m
 - si10
 - swvl1
 - swvl2
 - swvl3
 - stl2
 - stl3


The inspection confirms that all requested ERA5 variables were successfully downloaded and distributed across two NetCDF files.

The variables are organized into separate files according to their data characteristics, with precipitation and solar radiation stored independently from atmospheric and soil-related variables.

This step also identifies the abbreviated variable names used internally by ERA5. These names will be standardized in the next stage to improve readability and facilitate subsequent processing and analysis.


## ERA5 Dataset Loading and Integration

The climate variables are distributed across multiple NetCDF files.

In this step, the extracted files are loaded and merged into a single dataset containing all requested climate variables. This unified dataset serves as the foundation for subsequent climate processing and feature engineering.

In [14]:
# Load and merge ERA5 NetCDF files

datasets = []

for file in nc_files:

    ds_part = xr.open_dataset(
        file,
        engine="netcdf4"
    )

    if "valid_time" in ds_part.dims:
        ds_part = ds_part.rename(
            {"valid_time": "time"}
        )

    datasets.append(ds_part)

# Merge all ERA5 datasets into a single xarray dataset
ds_era5 = xr.merge(
    datasets,
    join="outer",
    compat="no_conflicts"
)

print("ERA5 variables:")

for var in ds_era5.data_vars:
    print(f" - {var}")

ERA5 variables:
 - tp
 - ssr
 - t2m
 - d2m
 - si10
 - swvl1
 - swvl2
 - swvl3
 - stl2
 - stl3


The successful merge provides a unified climate dataset containing all requested ERA5 variables. This consolidated structure simplifies subsequent processing and feature engineering step

## ERA5 Variable Standardization

ERA5 uses abbreviated variable names that are efficient for storage but may be difficult to interpret. To improve readability throughout the project, the variables are renamed using descriptive names:

| ERA5 Code | Description |
|------------|------------|
| t2m | 2m Air Temperature |
| d2m | 2m Dew Point Temperature |
| tp | Total Precipitation |
| ssr | Surface Net Solar Radiation |
| si10 | 10m Wind Speed |
| swvl1 | Soil Moisture Layer 1 |
| swvl2 | Soil Moisture Layer 2 |
| swvl3 | Soil Moisture Layer 3 |
| stl2 | Soil Temperature Layer 2 |
| stl3 | Soil Temperature Layer 3 |

In [15]:
# Rename ERA5 variables

ds_era5 = ds_era5.rename({

    "t2m": "Temperature_K",
    "d2m": "Dewpoint_Temperature_K",
    "si10": "Wind_Speed_10m_m_s",

    "tp": "Total_Precipitation_m",

    "ssr": "Surface_Net_Solar_Radiation_J_m2",

    "swvl1": "Soil_Moisture_Layer1",
    "swvl2": "Soil_Moisture_Layer2",
    "swvl3": "Soil_Moisture_Layer3",

    "stl2": "Soil_Temperature_Layer2_K",
    "stl3": "Soil_Temperature_Layer3_K"
})


print("ERA5 variables after renaming:")

for var in ds_era5.data_vars:
    print(f" - {var}")

ERA5 variables after renaming:
 - Total_Precipitation_m
 - Surface_Net_Solar_Radiation_J_m2
 - Temperature_K
 - Dewpoint_Temperature_K
 - Wind_Speed_10m_m_s
 - Soil_Moisture_Layer1
 - Soil_Moisture_Layer2
 - Soil_Moisture_Layer3
 - Soil_Temperature_Layer2_K
 - Soil_Temperature_Layer3_K


Standardizing the variable names improves readability and facilitates subsequent processing, analysis, and modeling activities while preserving the original ERA5 climate information.

## ERA5 Dataset Validation

Before proceeding to climate extraction and aggregation, the merged ERA5 dataset is validated to verify its structure, dimensions, coordinates, and available climate variables.

In [16]:
# Display dataset structures and variable names
print("Dataset dimensions:")
for dim, size in ds_era5.sizes.items():
    print(f" - {dim}: {size}")

print("\nAvailable climate variables:")
for var in ds_era5.data_vars:
    print(f" - {var}")

Dataset dimensions:
 - time: 960
 - latitude: 85
 - longitude: 237

Available climate variables:
 - Total_Precipitation_m
 - Surface_Net_Solar_Radiation_J_m2
 - Temperature_K
 - Dewpoint_Temperature_K
 - Wind_Speed_10m_m_s
 - Soil_Moisture_Layer1
 - Soil_Moisture_Layer2
 - Soil_Moisture_Layer3
 - Soil_Temperature_Layer2_K
 - Soil_Temperature_Layer3_K


The validation confirms that all requested climate variables are present in the merged ERA5 dataset.  
The data remains in its original gridded spatial format and is ready for country-level extraction and aggregation.

## ERA5 Climate Extraction Methodology

ERA5 climate data are provided on a spatial grid covering the entire Mediterranean region.

Using a single coordinate for each country may not adequately represent the climatic conditions experienced across olive-growing areas. To improve representativeness, climate indicators are extracted from regional bounding boxes corresponding to the main olive-producing zones of each country.

For each country, climate variables are averaged across all ERA5 grid cells contained within the selected region. This approach provides a more robust characterization of the environmental conditions affecting olive production while maintaining a manageable level of complexity.

In [17]:
# Geographical zones adjusted to Mediterranean olive-growing basins
COUNTRY_REGIONS = {
    "Algeria": {"lat_min": 35.0, "lat_max": 37.0, "lon_min": -2.0, "lon_max": 8.5},
    "Egypt": {"lat_min": 29.5, "lat_max": 31.5, "lon_min": 25.0, "lon_max": 34.0},
    "Greece": {"lat_min": 34.5, "lat_max": 41.0, "lon_min": 21.0, "lon_max": 27.0},
    "Italy": {"lat_min": 36.5, "lat_max": 43.0, "lon_min": 12.0, "lon_max": 18.5},
    "Libya": {"lat_min": 31.5, "lat_max": 33.0, "lon_min": 11.5, "lon_max": 15.0},
    "Morocco": {"lat_min": 32.0, "lat_max": 36.0, "lon_min": -9.5, "lon_max": -2.0},
    "Portugal": {"lat_min": 37.0, "lat_max": 42.0, "lon_min": -9.0, "lon_max": -6.5},
    "Spain": {"lat_min": 36.0, "lat_max": 41.5, "lon_min": -8.0, "lon_max": -1.0},
    "Syria": {"lat_min": 34.5, "lat_max": 37.0, "lon_min": 35.5, "lon_max": 38.0},
    "Tunisia": {"lat_min": 34.0, "lat_max": 37.5, "lon_min": 8.0, "lon_max": 11.5},
    "Turkiye": {"lat_min": 36.0, "lat_max": 41.0, "lon_min": 26.0, "lon_max": 37.0}  
}

GRID_RESOLUTION = 0.25 
VALID_OLIVE_POINTS = {}

for country, bounds in COUNTRY_REGIONS.items():
    # Adding + 0.01 prevents np.arange from cutting off the maximum bound due to floating-point rounding
    lat_range = np.arange(bounds["lat_min"], bounds["lat_max"] + 0.01, GRID_RESOLUTION)
    lon_range = np.arange(bounds["lon_min"], bounds["lon_max"] + 0.01, GRID_RESOLUTION)
    
    country_points = []
    
    for lat in lat_range:
        for lon in lon_range:

            # 1. TECHNICAL FIX: Keep only LAND (without 'not') and directly pass 'lon'
            if globe.is_land(lat, lon):

                # 2. LOGICAL FIX: Exclude deep desert by skipping latitudes that are too low
                if country == "Egypt" and lat < 30.0: 
                    continue  
                if country == "Algeria" and lat < 35.5: 
                    continue 
                if country == "Tunisia" and lat < 34.5: 
                    continue  
                if country == "Libya" and lat < 32.0:
                    continue
                if country == "Morocco" and lat < 33.0: 
                    continue

                country_points.append({"lat": round(float(lat), 4), "lon": round(float(lon), 4)})
                
    VALID_OLIVE_POINTS[country] = country_points
    
    print(f"🌍 {country}: {len(country_points)} olive coordinates retained on land.")

🌍 Algeria: 197 olive coordinates retained on land.
🌍 Egypt: 208 olive coordinates retained on land.
🌍 Greece: 221 olive coordinates retained on land.
🌍 Italy: 223 olive coordinates retained on land.
🌍 Libya: 55 olive coordinates retained on land.
🌍 Morocco: 206 olive coordinates retained on land.
🌍 Portugal: 203 olive coordinates retained on land.
🌍 Spain: 560 olive coordinates retained on land.
🌍 Syria: 100 olive coordinates retained on land.
🌍 Tunisia: 125 olive coordinates retained on land.
🌍 Turkiye: 760 olive coordinates retained on land.


The extraction methodology ensures that climate indicators are derived from regions that are more representative of olive-growing environments than a single-country centroid.

Land filtering and targeted exclusions reduce the influence of non-agricultural areas, particularly desert regions where climatic conditions are not representative of olive cultivation.

The retained coordinates will be used to compute country-level climate indicators from the ERA5 dataset.

## ERA5 Bronze Dataset Creation

The ERA5 climate data are currently stored as gridded observations covering the Mediterranean region.

In this step, climate variables are extracted from the predefined olive-producing regions and averaged across all ERA5 grid cells contained within each region.

The resulting Bronze dataset preserves the original monthly temporal resolution and retains the original ERA5 units, ensuring that the data remain as close as possible to the source while being organized in a tabular structure suitable for further processing.

In [18]:
# ERA5 Bronze monthly dataset

era5_blocks = []

# Extract climate data for each country
for country, points in VALID_OLIVE_POINTS.items():

    if not points:
        print(f"Warning: No valid coordinates found for {country}.")
        continue

    # Extract valid coordinates
    lats = [pt["lat"] for pt in points]
    lons = [pt["lon"] for pt in points]

    # Create point-based coordinate arrays for ERA5 extraction
    lat_idx = xr.DataArray(lats, dims="points")
    lon_idx = xr.DataArray(lons, dims="points")

    # Extract and spatially average ERA5 observations
    sub = ds_era5.sel(
        latitude=lat_idx,
        longitude=lon_idx,
        method="nearest"
    ).mean(dim="points")

    # Convert to dataframe
    sub_df = sub.to_dataframe().reset_index()

    # Create time features
    sub_df["time"] = pd.to_datetime(sub_df["time"])

    sub_df["Country"] = country
    sub_df["Year"] = sub_df["time"].dt.year
    sub_df["Month"] = sub_df["time"].dt.month

    # Build a unique Country-Year-Month structure
    sub_df = (
        sub_df
        .groupby(
            ["Country", "Year", "Month"],
            as_index=False
        )
        .agg({
            "Temperature_K": "mean",
            "Dewpoint_Temperature_K": "mean",
            "Wind_Speed_10m_m_s": "mean",
            "Total_Precipitation_m": "mean",
            "Surface_Net_Solar_Radiation_J_m2": "mean",
            "Soil_Moisture_Layer1": "mean",
            "Soil_Moisture_Layer2": "mean",
            "Soil_Moisture_Layer3": "mean",
            "Soil_Temperature_Layer2_K": "mean",
            "Soil_Temperature_Layer3_K": "mean"
        })
    )

    era5_blocks.append(sub_df)

if not era5_blocks:
    raise ValueError(
        "No ERA5 climate observations were extracted."
    )

# Concatenate all countries
era5_bronze = pd.concat(
    era5_blocks,
    ignore_index=True
)

# Sort observations
era5_bronze = (
    era5_bronze
    .sort_values(
        ["Country", "Year", "Month"]
    )
    .reset_index(drop=True)
)

# Export Bronze dataset
era5_bronze.to_csv(
    BRONZE_DIR / "era5_olive_bronze_monthly.csv",
    index=False,
    encoding="utf-8"
)

# Validation
print(f"ERA5 Bronze dataset shape: {era5_bronze.shape}")

print(f"\nDuplicate Country-Year-Month rows: {era5_bronze.duplicated(['Country', 'Year', 'Month']).sum()}")

print("\nMissing values:")
print(era5_bronze.isna().sum())

display(era5_bronze.head())

ERA5 Bronze dataset shape: (5280, 13)

Duplicate Country-Year-Month rows: 0

Missing values:
Country                             0
Year                                0
Month                               0
Temperature_K                       0
Dewpoint_Temperature_K              0
Wind_Speed_10m_m_s                  0
Total_Precipitation_m               0
Surface_Net_Solar_Radiation_J_m2    0
Soil_Moisture_Layer1                0
Soil_Moisture_Layer2                0
Soil_Moisture_Layer3                0
Soil_Temperature_Layer2_K           0
Soil_Temperature_Layer3_K           0
dtype: int64


,Country,Year,Month,Temperature_K,Dewpoint_Temperature_K,Wind_Speed_10m_m_s,Total_Precipitation_m,Surface_Net_Solar_Radiation_J_m2,Soil_Moisture_Layer1,Soil_Moisture_Layer2,Soil_Moisture_Layer3,Soil_Temperature_Layer2_K,Soil_Temperature_Layer3_K
0,Algeria,1985,1,279.529510,275.559570,2.866066,0.002221,7424540.5,0.333171,0.327917,0.280086,279.784790,281.430481
1,Algeria,1985,2,284.470215,278.602295,2.753909,0.001231,11236066.0,0.282179,0.298915,0.279841,283.254822,283.194641
2,Algeria,1985,3,282.166626,277.736572,2.864231,0.003723,12607504.0,0.313997,0.318074,0.292551,282.467468,282.791229
3,Algeria,1985,4,287.549072,280.234009,2.788835,0.001368,18208682.0,0.246970,0.281292,0.287363,286.934204,285.623138
4,Algeria,1985,5,289.547821,283.370300,2.586975,0.002478,17645158.0,0.275779,0.291037,0.275117,289.074066,287.716095


The validation confirms that the climate extraction methodology successfully generated a complete monthly dataset for all selected countries.

Each observation represents a unique Country-Year-Month combination, ensuring consistency with the intended analytical structure.

The absence of missing values indicates that all climate variables were successfully extracted from the selected olive-growing regions, providing a complete climate record for subsequent aggregation and feature engineering in the Silver layer.


## Export ERA5 Bronze Dataset

The validated ERA5 Bronze dataset is exported to the project's Bronze layer to ensure reproducibility and support downstream processing within the Medallion Architecture pipeline.

In [19]:
# Export Bronze dataset
era5_bronze.to_csv(
    BRONZE_DIR / "era5_olive_bronze_monthly.csv",
    index=False,
    encoding="utf-8"
)

print("ERA5 Bronze dataset exported successfully.")

ERA5 Bronze dataset exported successfully.



## ERA5 Silver Dataset Creation

The ERA5 Bronze dataset contains monthly climate observations in their original ERA5 units.

In this step, climate variables are converted into analysis-friendly units and aggregated into annual Country-Year indicators.

The resulting ERA5 Silver dataset provides annual climate metrics suitable for integration with agricultural and socio-economic datasets.

In [22]:
# Create working copy
era5_silver_source = era5_bronze.copy()


# Convert temperature variables (K → °C) and rename the corresponding column
era5_silver_source["Temperature_K"] = (
    era5_silver_source["Temperature_K"] - 273.15
)
era5_silver_source = era5_silver_source.rename(columns={"Temperature_K": "Mean_Temperature_C"})


era5_silver_source["Dewpoint_Temperature_K"] = (
    era5_silver_source["Dewpoint_Temperature_K"] - 273.15
)
era5_silver_source = era5_silver_source.rename(columns={"Dewpoint_Temperature_K": "Mean_Dewpoint_C"})


era5_silver_source["Soil_Temperature_Layer2_K"] = (
    era5_silver_source["Soil_Temperature_Layer2_K"] - 273.15
)
era5_silver_source = era5_silver_source.rename(columns={"Soil_Temperature_Layer2_K": "Mean_Soil_Temperature_7_28cm_C"})


era5_silver_source["Soil_Temperature_Layer3_K"] = (
    era5_silver_source["Soil_Temperature_Layer3_K"] - 273.15
)
era5_silver_source = era5_silver_source.rename(columns={"Soil_Temperature_Layer3_K": "Mean_Soil_Temperature_28_100cm_C"})


# Convert precipitation (m → mm) and rename the corresponding column


era5_silver_source["Days_In_Month"] = pd.to_datetime(
    era5_silver_source["Year"].astype(str)
    + "-"
    + era5_silver_source["Month"].astype(str)
    + "-01"
).dt.days_in_month


era5_silver_source["Total_Precipitation_m"] = (
    era5_silver_source["Total_Precipitation_m"]
    * 1000
    * era5_silver_source["Days_In_Month"]
)
era5_silver_source = era5_silver_source.rename(columns={"Total_Precipitation_m": "Total_Precipitation_mm"})



# Convert solar radiation (J/m² → MJ/m²) and rename the corresponding column

era5_silver_source["Surface_Net_Solar_Radiation_J_m2"] = (
    (era5_silver_source["Surface_Net_Solar_Radiation_J_m2"] / 1_000_000) * era5_silver_source["Days_In_Month"]
)
era5_silver_source = era5_silver_source.rename(columns={"Surface_Net_Solar_Radiation_J_m2": "Total_Solar_Radiation_MJ_m2"})


# Remove temporary column
era5_silver_source = era5_silver_source.drop(columns=["Days_In_Month"])


# Rename remaining variables

era5_silver_source = era5_silver_source.rename(columns={
    "Wind_Speed_10m_m_s": "Mean_Wind_Speed_m_s",
    "Soil_Moisture_Layer1": "Mean_Soil_Moisture_0_7cm",
    "Soil_Moisture_Layer2": "Mean_Soil_Moisture_7_28cm",
    "Soil_Moisture_Layer3": "Mean_Soil_Moisture_28_100cm"
})



# Aggregate monthly observations to the Country-Year level

era5_silver = (
    era5_silver_source
    .groupby(["Country", "Year"], as_index=False)
    .agg({
        "Mean_Temperature_C": "mean",
        "Mean_Dewpoint_C": "mean",
        "Mean_Wind_Speed_m_s": "mean",
        "Total_Precipitation_mm": "sum",
        "Total_Solar_Radiation_MJ_m2": "sum",
        "Mean_Soil_Moisture_0_7cm": "mean",
        "Mean_Soil_Moisture_7_28cm": "mean",
        "Mean_Soil_Moisture_28_100cm": "mean",
        "Mean_Soil_Temperature_7_28cm_C": "mean",
        "Mean_Soil_Temperature_28_100cm_C": "mean"
    })
)

# Sort observations
era5_silver = (
    era5_silver
    .sort_values(["Country", "Year"])
    .reset_index(drop=True)
)


display(era5_silver.head())


,Country,Year,Mean_Temperature_C,Mean_Dewpoint_C,Mean_Wind_Speed_m_s,Total_Precipitation_mm,Total_Solar_Radiation_MJ_m2,Mean_Soil_Moisture_0_7cm,Mean_Soil_Moisture_7_28cm,Mean_Soil_Moisture_28_100cm,Mean_Soil_Temperature_7_28cm_C,Mean_Soil_Temperature_28_100cm_C
0,Algeria,1985,16.395411,8.906636,2.479213,542.142498,5237.748049,0.228522,0.247775,0.236782,16.382143,16.332598
1,Algeria,1986,16.141417,8.764529,2.551661,568.276248,5197.275744,0.231651,0.246896,0.230832,16.232656,16.280464
2,Algeria,1987,16.968359,9.029343,2.552873,452.002260,5238.071111,0.216728,0.237414,0.235118,16.946756,16.861189
3,Algeria,1988,16.701139,8.888471,2.489093,466.610421,5278.981691,0.207998,0.225206,0.204125,16.860826,16.951820
4,Algeria,1989,16.784948,9.079582,2.414711,390.857869,5358.641980,0.203087,0.225545,0.212328,16.751350,16.664587


The dataset includes annual measures of temperature, precipitation, humidity, wind speed, solar radiation, soil moisture, and soil temperature for all study countries.

The transformation aligns the climate data with the analytical structure of the FAOSTAT and World Bank datasets.

The resulting dataset preserves the key climatic information required for subsequent integration, analysis, and modeling.

## ERA5 Silver Dataset Validation

The monthly and annual ERA5 Silver datasets are validated to verify completeness, consistency, and uniqueness before integration with the agricultural and socio-economic datasets.

In [23]:
print(
    f"Monthly ERA5 Dataset Overview:\n"
    f" - Data Shape: {era5_silver_source.shape}\n"
    f" - Duplicate Rows (Country-Year-Month): {era5_silver_source.duplicated(['Country', 'Year', 'Month']).sum()}\n"
    f" - Total Missing Values: {era5_silver_source.isna().sum().sum()}\n\n"
    f"Annual ERA5 Dataset Overview:\n"
    f" - Data Shape: {era5_silver.shape}\n"
    f" - Duplicate Rows (Country-Year): {era5_silver.duplicated(['Country', 'Year']).sum()}\n"
    f" - Total Missing Values: {era5_silver.isna().sum().sum()}"
)

Monthly ERA5 Dataset Overview:
 - Data Shape: (5280, 13)
 - Duplicate Rows (Country-Year-Month): 0
 - Total Missing Values: 0

Annual ERA5 Dataset Overview:
 - Data Shape: (440, 12)
 - Duplicate Rows (Country-Year): 0
 - Total Missing Values: 0


## Export ERA5 Silver Datasets

The validated monthly and annual ERA5 Silver datasets are exported for downstream integration and analysis.

In [24]:
era5_silver_source.to_csv(
    SILVER_DIR / "era5_olive_silver_monthly.csv",
    index=False,
    encoding="utf-8"
)

era5_silver.to_csv(
    SILVER_DIR / "era5_olive_silver_yearly.csv",
    index=False,
    encoding="utf-8"
)

print("ERA5 Silver datasets exported successfully.")

ERA5 Silver datasets exported successfully.


## World Bank Data Collection

To complement the agricultural and climate datasets, socio-economic indicators are retrieved from the World Bank Open Data API.

The selected indicators capture demographic, economic, and land-use characteristics that may influence olive production systems across the study countries.

Although climate data are available from 1985 onwards, the socio-economic analysis is restricted to the period **2000–2024** due to substantial missing data for several World Bank indicators prior to 2000. Restricting the study period improves data completeness and ensures consistency across the selected indicators.

The downloaded data will form the socio-economic component of the Bronze layer and will later be transformed into a standardized Country-Year structure.

### Selected Indicators: 

- Population
- GDP per capita (current US$)
- Agricultural land (% of land area)
- Forest area (% of land area)
- Rural population (% of total population)
- Agriculture value added (% of GDP)

In [25]:
WB_INDICATORS = {
    'SP.POP.TOTL': 'Population',
    'NY.GDP.PCAP.CD': 'GDP_per_capita',
    'AG.LND.AGRI.ZS': 'Agricultural_land_pct',
    'AG.LND.FRST.ZS': 'Forest_area_pct',
    'SP.RUR.TOTL.ZS': 'Rural_population_pct',
    'NV.AGR.TOTL.ZS': 'Agriculture_value_added_pct_GDP'
}

def fetch_worldbank_indicator(
    country_codes,
    indicator_code,
    indicator_name,
    start_year=2000,
    end_year=2024
):

    country_str = ';'.join(country_codes)

    url = (
        f'https://api.worldbank.org/v2/country/{country_str}/indicator/{indicator_code}'
        f'?format=json&date={start_year}:{end_year}&per_page=20000'
    )

    response = requests.get(url, timeout=60)
    response.raise_for_status()
    payload = response.json()

    if len(payload) < 2 or payload[1] is None:
        return pd.DataFrame(columns=['Country', 'Country_Code', 'Year', indicator_name])

    rows = []
    for item in payload[1]:
        code = item['countryiso3code']
        rows.append({
            'Country': COUNTRIES_ISO3.get(code, item['country']['value']),
            'Country_Code': code,
            'Year': int(item['date']),
            indicator_name: item['value']
        })

    return pd.DataFrame(rows)


print(
    f"Configured {len(WB_INDICATORS)} World Bank indicators "
    "and initialized the data retrieval function."
)


Configured 6 World Bank indicators and initialized the data retrieval function.


## World Bank Bronze Dataset Creation

The selected World Bank indicators are retrieved for each study country and stored in a long-format structure.

At this stage, the data remains close to its original source format, with each observation representing a specific indicator for a given country and year.

The resulting dataset constitutes the socio-economic Bronze layer and will later be transformed into a standardized Country-Year structure.

In [26]:
worldbank_long_parts = []

for indicator_code, indicator_name in WB_INDICATORS.items():

    print(f"Downloading: {indicator_name}")

    df_indicator = fetch_worldbank_indicator(
        country_codes=list(COUNTRIES_ISO3.keys()),
        indicator_code=indicator_code,
        indicator_name=indicator_name,
        start_year=2000,
        end_year=2024
    )

    df_indicator["Indicator_Code"] = indicator_code
    df_indicator["Indicator_Name"] = indicator_name
    df_indicator["Value"] = df_indicator[indicator_name]

    worldbank_long_parts.append(
        df_indicator[
            [
                "Country",
                "Country_Code",
                "Year",
                "Indicator_Code",
                "Indicator_Name",
                "Value"
            ]
        ]
    )

worldbank_bronze = pd.concat(
    worldbank_long_parts,
    ignore_index=True
)

print(f"World Bank Bronze dataset shape: {worldbank_bronze.shape}")

display(worldbank_bronze.head())

Downloading: Population
Downloading: GDP_per_capita
Downloading: Agricultural_land_pct
Downloading: Forest_area_pct
Downloading: Rural_population_pct
Downloading: Agriculture_value_added_pct_GDP
World Bank Bronze dataset shape: (1650, 6)


,Country,Country_Code,Year,Indicator_Code,Indicator_Name,Value
0,Algeria,DZA,2024,SP.POP.TOTL,Population,46814308.0
1,Algeria,DZA,2023,SP.POP.TOTL,Population,46164219.0
2,Algeria,DZA,2022,SP.POP.TOTL,Population,45477389.0
3,Algeria,DZA,2021,SP.POP.TOTL,Population,44761099.0
4,Algeria,DZA,2020,SP.POP.TOTL,Population,44042091.0


The selected World Bank indicators were successfully downloaded and combined into a single long-format dataset.  
The resulting Bronze dataset provides the socio-economic component of the project while remaining close to the original structure of the source data.  
Each observation represents a specific indicator, country, and year combination, ensuring traceability to the original World Bank records.  
The dataset is ready for validation and subsequent transformation into the Silver layer.

## World Bank Bronze Dataset Validation

The World Bank Bronze dataset is validated to verify its completeness, structure, and consistency before any transformation is performed.

The objective of this section is to ensure that the downloaded indicators cover the expected countries and years and that the resulting dataset is suitable for subsequent Silver-layer processing.


In [27]:
print(
    f"Dataset shape: {worldbank_bronze.shape}\n"
    f"\nColumns:\n{worldbank_bronze.columns.tolist()}\n"
    f"\nMissing values:\n{worldbank_bronze.isna().sum().sort_values(ascending=False)}\n"
    f"\nDuplicate rows: {worldbank_bronze.duplicated().sum()}\n"
    f"\nCountries included:\n{sorted(worldbank_bronze['Country'].dropna().unique())}\n"
    f"\nYears covered: {worldbank_bronze['Year'].min()} - {worldbank_bronze['Year'].max()}\n"
    f"\nIndicators included:\n{sorted(worldbank_bronze['Indicator_Name'].dropna().unique())}"
)

Dataset shape: (1650, 6)

Columns:
['Country', 'Country_Code', 'Year', 'Indicator_Code', 'Indicator_Name', 'Value']

Missing values:
Value             28
Country            0
Country_Code       0
Year               0
Indicator_Code     0
Indicator_Name     0
dtype: int64

Duplicate rows: 0

Countries included:
['Algeria', 'Egypt', 'Greece', 'Italy', 'Libya', 'Morocco', 'Portugal', 'Spain', 'Syria', 'Tunisia', 'Turkiye']

Years covered: 2000 - 2024

Indicators included:
['Agricultural_land_pct', 'Agriculture_value_added_pct_GDP', 'Forest_area_pct', 'GDP_per_capita', 'Population', 'Rural_population_pct']


The validation confirms that the data extraction process was successful and that all selected indicators, countries, and years are represented in the dataset.

Missing values are concentrated in a limited number of indicators and years, primarily affecting recent observations and a small number of records for specific countries. These missing values originate from the World Bank source data and will be addressed during the Silver transformation stage.

The Bronze dataset is suitable for subsequent processing and transformation.

## Export World Bank Bronze Dataset

The validated World Bank Bronze dataset is exported to the Bronze layer for downstream processing.


In [28]:
worldbank_bronze.to_csv(
    BRONZE_DIR / "worldbank_olive_bronze.csv",
    index=False,
    encoding="utf-8"
)

print("World Bank Bronze dataset exported successfully!")

World Bank Bronze dataset exported successfully!


## World Bank Silver Dataset Creation

The World Bank Bronze dataset preserves the original indicator-oriented structure, where each observation corresponds to a specific indicator, country, and year.

To facilitate integration with the agricultural and climate datasets, the data is transformed into a standardized Country-Year format. Each selected indicator becomes a separate column, resulting in a single record per country and year.

The resulting dataset constitutes the World Bank Silver layer and is aligned with the analytical structure used throughout the project.


In [29]:
# Transform indicators into a Country-Year structure

worldbank_silver = (
    worldbank_bronze
    .pivot_table(
        index=[
            "Country",
            "Country_Code",
            "Year"
        ],
        columns="Indicator_Name",
        values="Value",
        aggfunc="first"
    )
    .reset_index()
)

worldbank_silver.columns.name = None

# Sort observations
worldbank_silver = (
    worldbank_silver
    .sort_values(
        ["Country", "Year"]
    )
    .reset_index(drop=True)
)

print(
    f"World Bank Silver dataset shape: "
    f"{worldbank_silver.shape}"
)

display(worldbank_silver.head())


World Bank Silver dataset shape: (275, 9)


,Country,Country_Code,Year,Agricultural_land_pct,Agriculture_value_added_pct_GDP,Forest_area_pct,GDP_per_capita,Population,Rural_population_pct
0,Algeria,DZA,2000,16.803261,8.395048,0.662961,1772.928691,30903893.0,40.147876
1,Algeria,DZA,2001,16.840209,8.886180,0.677194,1896.300209,31331221.0,39.382283
2,Algeria,DZA,2002,16.733565,8.413026,0.691427,1937.464114,31750835.0,38.618734
3,Algeria,DZA,2003,16.754851,8.456720,0.705661,2283.772993,32175818.0,37.857209
4,Algeria,DZA,2004,17.275185,7.722174,0.719894,2816.993850,32628286.0,37.097687


The transformation aligns the World Bank data with the structure of the FAOSTAT and ERA5 Silver datasets.  
This standardized format simplifies dataset integration and supports subsequent analytical and modeling activities.

## World Bank Silver Dataset Validation

This section is a validation section to verify the completeness, consistency fo the dataset the and Country-Year structure before integration with the agricultural and climate datasets.

In [30]:
import pandas as pd

# Sort data sequentially to ensure accurate chronological filling
worldbank_silver = worldbank_silver.sort_values(["Country", "Year"])

# Apply forward-fill only to slow-changing structural indicators
stable_indicators = ["Agricultural_land_pct", "Forest_area_pct"]
for col in stable_indicators:
    worldbank_silver[col] = worldbank_silver.groupby("Country")[col].ffill()

# Dataset Validation Overview
print(
    f"# Dataset Validation Overview\n"
    f"World Bank Silver Dataset Validation:\n"
    f" - Dataset Shape: {worldbank_silver.shape}\n"
    f" - Temporal Coverage: {worldbank_silver['Year'].min()} - {worldbank_silver['Year'].max()}\n"
    f" - Total Countries: {worldbank_silver['Country'].nunique()}\n"
    f" - Duplicate Rows (Country-Year): {worldbank_silver.duplicated(['Country', 'Year']).sum()}"
)

# Inspect remaining missing values (expected for economic indicators)
print("\nRemaining missing values by column:")
print(worldbank_silver.isna().sum()[worldbank_silver.isna().sum() > 0])


# Dataset Validation Overview
World Bank Silver Dataset Validation:
 - Dataset Shape: (275, 9)
 - Temporal Coverage: 2000 - 2024
 - Total Countries: 11
 - Duplicate Rows (Country-Year): 0

Remaining missing values by column:
Agriculture_value_added_pct_GDP    4
GDP_per_capita                     2
dtype: int64


The World Bank Silver transformation successfully consolidated the selected indicators into a standardized Country-Year structure.  
The dataset contains 275 observations, 9 variables and extends over 25 years (2000–2024) for 11 countries.No duplicated results were recorded.  

Recent structural gaps in Agricultural_land_pct and Forest_area_pct caused by World Bank API updates were resolved using a targeted forward-fill.  

Two missing values on GDP_per_capita and 4 in Agriculture_value_added_pct_GDP were intentionally retained to avoid introducing scientific bias.  
The dataset is ready for integration with the FAOSTAT and ERA5 Silver datasets.


## Export World Bank Silver Dataset

The validated World Bank Silver dataset is exported for downstream integration and analysis.


In [31]:
worldbank_silver.to_csv(
    SILVER_DIR / "worldbank_olive_silver.csv",
    index=False,
    encoding="utf-8"
)

print("World Bank Silver dataset exported successfully.")

World Bank Silver dataset exported successfully.


## Silver Dataset Integration

In this step, the FAOSTAT, ERA5, and World Bank Silver datasets are integrated into a single analytical dataset combining agricultural production, climatic conditions, and socio-economic characteristics for each country and year.  
The resulting integrated dataset serves as the foundation for feature engineering, exploratory analysis, dashboard development, and predictive modeling.

In [32]:
# Integrate Silver datasets

integrated_silver = (
    faostat_silver
    .merge(
        era5_silver,
        on=["Country", "Year"],
        how="inner"
    )
    .merge(
        worldbank_silver.drop(columns=["Country_Code"]),
        on=["Country", "Year"],
        how="inner"
    )
)

# Sort observations
integrated_silver = (
    integrated_silver
    .sort_values(
        ["Country", "Year"]
    )
    .reset_index(drop=True)
)

print(f"Integrated Silver dataset shape: {integrated_silver.shape}")

display(integrated_silver.head())

Integrated Silver dataset shape: (274, 21)


,Country,Year,Area_Harvested_ha,Production_tonnes,Yield_kg_per_ha,Mean_Temperature_C,Mean_Dewpoint_C,Mean_Wind_Speed_m_s,Total_Precipitation_mm,Total_Solar_Radiation_MJ_m2,...,Mean_Soil_Moisture_7_28cm,Mean_Soil_Moisture_28_100cm,Mean_Soil_Temperature_7_28cm_C,Mean_Soil_Temperature_28_100cm_C,Agricultural_land_pct,Agriculture_value_added_pct_GDP,Forest_area_pct,GDP_per_capita,Population,Rural_population_pct
0,Algeria,2000,168080.0,217112.0,1291.7,17.070543,8.165996,2.534910,311.594795,5548.199982,...,0.211491,0.202367,16.958441,16.968164,16.803261,8.395048,0.662961,1772.928691,30903893.0,40.147876
1,Algeria,2001,177220.0,200339.0,1130.5,17.351156,8.849744,2.526347,400.795290,5427.257452,...,0.217278,0.199521,17.315872,17.335070,16.840209,8.886180,0.677194,1896.300209,31331221.0,39.382283
2,Algeria,2002,190550.0,191926.0,1007.2,17.139296,8.822309,2.611674,422.704252,5404.462550,...,0.208697,0.177572,16.973539,16.920982,16.733565,8.413026,0.691427,1937.464114,31750835.0,38.618734
3,Algeria,2003,209730.0,167627.0,799.3,17.291044,9.750931,2.598855,679.290588,5150.412796,...,0.247167,0.222747,17.153269,17.197523,16.754851,8.456720,0.705661,2283.772993,32175818.0,37.857209
4,Algeria,2004,226337.0,468800.0,2071.2,16.630468,9.407756,2.540032,596.052643,5214.946519,...,0.240216,0.220955,16.403954,16.449976,17.275185,7.722174,0.719894,2816.993850,32628286.0,37.097687


The integration creates a comprehensive analytical dataset that captures multiple dimensions influencing olive production across Mediterranean countries. Each observation combines agricultural, climate, and socio-economic information for a specific country and year.

The resulting dataset provides a unified foundation for feature engineering, exploratory analysis, and predictive modeling.

## Integrated Silver Dataset Validation

The integrated Silver dataset is validated to verify the success of the merge operations and assess data completeness before feature engineering and Gold-layer creation.

In [33]:
# Validate integrated Silver dataset
print(
    f"Dataset shape: {integrated_silver.shape}\n"
    f"\nMissing values:\n{integrated_silver.isna().sum().sort_values(ascending=False)}\n"
    f"\nDuplicate Country-Year rows: {integrated_silver.duplicated(['Country', 'Year']).sum()}\n"
    f"\nCountries included: {integrated_silver['Country'].nunique()}\n"
    f"\nYears covered: {integrated_silver['Year'].min()} - {integrated_silver['Year'].max()}"
)

Dataset shape: (274, 21)

Missing values:
Agriculture_value_added_pct_GDP     4
GDP_per_capita                      2
Area_Harvested_ha                   0
Year                                0
Country                             0
Yield_kg_per_ha                     0
Production_tonnes                   0
Mean_Temperature_C                  0
Mean_Dewpoint_C                     0
Total_Solar_Radiation_MJ_m2         0
Mean_Soil_Moisture_0_7cm            0
Mean_Wind_Speed_m_s                 0
Total_Precipitation_mm              0
Mean_Soil_Moisture_28_100cm         0
Mean_Soil_Moisture_7_28cm           0
Mean_Soil_Temperature_28_100cm_C    0
Mean_Soil_Temperature_7_28cm_C      0
Agricultural_land_pct               0
Forest_area_pct                     0
Population                          0
Rural_population_pct                0
dtype: int64

Duplicate Country-Year rows: 0

Countries included: 11

Years covered: 2000 - 2024


The integration process successfully aligned the agricultural, climate, and socio-economic datasets using a common Country-Year structure. it produced a unified analytical dataset with: 274 observations, 21 variables, 11 studied countries and covered the period from 2000 to 2024.  

No duplicate observations were introduced during the merge process, indicating that the three Silver datasets were combined consistently.  

The missing values originated from the original source datasets rather than the integration procedure itself. Their number is small relative to the overall dataset size and therefore has a limited impact on dataset completeness.  

All remaining variables contain complete information and the integrated Silver dataset is ready for Gold-layer feature engineering.  

## Export Integrated Silver Dataset

The validated integrated Silver dataset is exported for feature engineering and Gold-layer dataset creation.

In [34]:
integrated_silver.to_csv(
    SILVER_DIR / "integrated_olive_silver.csv",
    index=False,
    encoding="utf-8"
)

print("Integrated Silver dataset exported successfully.")

Integrated Silver dataset exported successfully.



## Temperature Change Enrichment

To enrich the analytical dataset with long-term climate dynamics, annual temperature change was retrieved from the FAOSTAT Environmental Indicators bulk dataset.


In [35]:
# FAOSTAT Temperature Change Dataset Download
TC_BULK_URL = (
    "https://bulks-faostat.fao.org/production/"
    "Environment_Temperature_change_E_All_Data_(Normalized).zip"
)

response = requests.get(TC_BULK_URL, timeout=60)
response.raise_for_status()

with ZipFile(BytesIO(response.content)) as z:
    temperature_change_source = pd.read_csv(
        z.open("Environment_Temperature_change_E_All_Data_(Normalized).csv"),
        low_memory=False,
    )

# Standardize country names and filter for study scope
temperature_change_source["Area"] = temperature_change_source["Area"].apply(
    normalize_country_name
)
countries = list(COUNTRIES_ISO3.values())

# Extract Annual Temperature Change
temperature_change_annual = (
    temperature_change_source.loc[
        temperature_change_source["Area"].isin(countries)
        & temperature_change_source["Months"].eq("Meteorological year")
        & temperature_change_source["Element"].eq("Temperature change"),
        ["Area", "Year", "Element", "Value"],
    ]
    .pivot(index=["Area", "Year"], columns="Element", values="Value")
    .reset_index()
    .rename(
        columns={"Area": "Country", "Temperature change": "Temperature_change"}
    )
    .sort_values(["Country", "Year"])
    .reset_index(drop=True)
)

print(
    f"Annual temperature change dataset successfully created. Shape: {temperature_change_annual.shape}"
)


Annual temperature change dataset successfully created. Shape: (715, 3)


## Temperature Change Dataset Validation

The generated temperature change datasets are validated to verify their completeness, consistency, and suitability for integration into the analytical dataset.

In [36]:
# Validate Annual Temperature Change Dataset
print(f"Annual Dataset Shape: {temperature_change_annual.shape}")
print(f" - Temporal Coverage: {temperature_change_annual['Year'].min()} - {temperature_change_annual['Year'].max()}")
print(f" - Total Countries: {temperature_change_annual['Country'].nunique()}")
print(f" - Duplicate Rows (Country-Year): {temperature_change_annual.duplicated(['Country', 'Year']).sum()}")

# Check for missing temperature entries
missing_values = temperature_change_annual["Temperature_change"].isna().sum()
print(f" - Missing Temperature Change Values: {missing_values}")


Annual Dataset Shape: (715, 3)
 - Temporal Coverage: 1961 - 2025
 - Total Countries: 11
 - Duplicate Rows (Country-Year): 0
 - Missing Temperature Change Values: 0


## Integrated Dataset Enrichment

The annual temperature change indicator is integrated into the analytical dataset to complement the existing agricultural, climatic, and socio-economic variables.

In [37]:
integrated_silver_enriched = (
    integrated_silver
    .merge(
        temperature_change_annual,
        on=["Country", "Year"],
        how="inner"
    )
)

print(f"Enriched Silver dataset shape: {integrated_silver_enriched.shape}")

display(integrated_silver_enriched.head())

Enriched Silver dataset shape: (274, 22)


,Country,Year,Area_Harvested_ha,Production_tonnes,Yield_kg_per_ha,Mean_Temperature_C,Mean_Dewpoint_C,Mean_Wind_Speed_m_s,Total_Precipitation_mm,Total_Solar_Radiation_MJ_m2,...,Mean_Soil_Moisture_28_100cm,Mean_Soil_Temperature_7_28cm_C,Mean_Soil_Temperature_28_100cm_C,Agricultural_land_pct,Agriculture_value_added_pct_GDP,Forest_area_pct,GDP_per_capita,Population,Rural_population_pct,Temperature_change
0,Algeria,2000,168080.0,217112.0,1291.7,17.070543,8.165996,2.534910,311.594795,5548.199982,...,0.202367,16.958441,16.968164,16.803261,8.395048,0.662961,1772.928691,30903893.0,40.147876,0.765
1,Algeria,2001,177220.0,200339.0,1130.5,17.351156,8.849744,2.526347,400.795290,5427.257452,...,0.199521,17.315872,17.335070,16.840209,8.886180,0.677194,1896.300209,31331221.0,39.382283,1.793
2,Algeria,2002,190550.0,191926.0,1007.2,17.139296,8.822309,2.611674,422.704252,5404.462550,...,0.177572,16.973539,16.920982,16.733565,8.413026,0.691427,1937.464114,31750835.0,38.618734,1.187
3,Algeria,2003,209730.0,167627.0,799.3,17.291044,9.750931,2.598855,679.290588,5150.412796,...,0.222747,17.153269,17.197523,16.754851,8.456720,0.705661,2283.772993,32175818.0,37.857209,1.518
4,Algeria,2004,226337.0,468800.0,2071.2,16.630468,9.407756,2.540032,596.052643,5214.946519,...,0.220955,16.403954,16.449976,17.275185,7.722174,0.719894,2816.993850,32628286.0,37.097687,0.920


The annual temperature change indicator was successfully integrated into the analytical dataset.

The enriched dataset combines agricultural production, climate indicators, socio-economic information, and long-term temperature trends within a unified Country-Year framework.


## Enriched Silver Dataset Validation

The enriched Silver dataset is validated to verify the successful integration of the temperature change indicator and assess data completeness prior to Gold-layer feature engineering.


In [38]:
print(
    f"Dataset shape: {integrated_silver_enriched.shape}\n"
    f"\nMissing values:\n{integrated_silver_enriched.isna().sum().sort_values(ascending=False)}\n"
    f"\nDuplicate Country-Year rows: {integrated_silver_enriched.duplicated(['Country', 'Year']).sum()}\n"
    f"\nCountries included: {integrated_silver_enriched['Country'].nunique()}\n"
    f"\nYears covered: {integrated_silver_enriched['Year'].min()} - {integrated_silver_enriched['Year'].max()}"
)

Dataset shape: (274, 22)

Missing values:
Agriculture_value_added_pct_GDP     4
GDP_per_capita                      2
Area_Harvested_ha                   0
Production_tonnes                   0
Country                             0
Year                                0
Mean_Temperature_C                  0
Yield_kg_per_ha                     0
Mean_Dewpoint_C                     0
Mean_Wind_Speed_m_s                 0
Mean_Soil_Moisture_0_7cm            0
Mean_Soil_Moisture_7_28cm           0
Total_Precipitation_mm              0
Total_Solar_Radiation_MJ_m2         0
Mean_Soil_Temperature_7_28cm_C      0
Mean_Soil_Moisture_28_100cm         0
Agricultural_land_pct               0
Mean_Soil_Temperature_28_100cm_C    0
Forest_area_pct                     0
Population                          0
Rural_population_pct                0
Temperature_change                  0
dtype: int64

Duplicate Country-Year rows: 0

Countries included: 11

Years covered: 2000 - 2024


The final dataset successfully consolidates all agricultural, climatic, and socio-economic variables into a clean, uniform Country-Year structure.  

The dataset contains 274 observations and 22 variables, extending over 25 years (2000–2024) for 11 countries, with zero duplicate rows recorded.  

The only remaining gaps are the 2 missing values for GDP_per_capita and 4 missing values for Agriculture_value_added_pct_GDP. These specific economic gaps were intentionally left untouched to prevent data bias and have no impact on the complete, fully available agricultural and climatic time series.

## Export Enriched Silver Dataset

The validated enriched dataset and supplemental temperature change datasets are exported for downstream analysis.

In [39]:
# Export clean annual datasets
temperature_change_annual.to_csv(
    SILVER_DIR / "temperature_change_annual.csv", index=False, encoding="utf-8"
)

integrated_silver_enriched.to_csv(
    SILVER_DIR / "integrated_silver_enriched.csv", index=False, encoding="utf-8"
)

print("Annual temperature change and enriched Silver datasets exported successfully.")


Annual temperature change and enriched Silver datasets exported successfully.


## Executive Summary

This notebook completed the Bronze and Silver stages of the data pipeline.

Agricultural data from FAOSTAT, climate indicators derived from ERA5, and socioeconomic and land-use indicators from the World Bank were collected, standardized, validated, and integrated at the Country–Year level. The consolidated dataset was then enriched with the annual FAOSTAT temperature-change indicator, expressed in °C relative to the **1951–1980 reference period**.

The final enriched Silver dataset contains:

- 274 Country–Year observations;
- 22 variables;
- 11 Mediterranean countries;
- data spanning 2000–2024;
- no duplicate Country–Year records.

Six socioeconomic values remain missing:

- 2 values for GDP_per_capita;
- 4 values for Agriculture_value_added_pct_GDP.

These values were retained as missing because imputation was not sufficiently supported by the available data. The agricultural and climate variables contain no missing values within the final dataset.

The validated enriched Silver dataset was exported as the primary input for Gold-layer feature engineering in Notebook 02.